# CS781 Phase 1 - Encoder-Decoder legal LM (1B-token, 2xT4, resumable)

**How this notebook runs**
- Use **Save Version -> Save & Run All (Commit)** with *GPU T4 x2*, so training keeps going even if the browser closes.
- Section 4-5 (download + tokenizer + tokenization) run once. Tokenizer, token files and checkpoints are written to `/kaggle/working`.
- Training (section 8) runs as a 2-GPU DDP job. After `SESSION_BUDGET_HOURS` it saves a checkpoint and stops cleanly; the evaluation cells then skip themselves.
- **To continue in a new session:** new version -> *Add Input -> Notebook Output Files -> this notebook* -> Run All. The tokenized corpus, tokenizer and newest checkpoint are found automatically (nothing is re-downloaded or re-tokenized) and the data order resumes exactly where it stopped.
- When the single pass over the corpus is finished, sections 9-10 evaluate the best checkpoint and build the submission zip.


# 1. Login

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

# 2. Check GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

# 3. Configuration

In [ ]:
import os
import gc
import math
import time
import glob
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# -----------------------------------------------------------------------------
# File path setup
# -----------------------------------------------------------------------------
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

DATA_DIR = "Phase_1_Pretraining"
TRAIN_DIR = os.path.join(DATA_DIR, "Train")
TEST_FILE = os.path.join(DATA_DIR, "Test", "test.json")

# Collect ALL training files inside the Train directory (not just Supreme Court)
if os.path.exists(TRAIN_DIR):
    TRAIN_FILES = sorted(glob.glob(os.path.join(TRAIN_DIR, "*.json")))
else:
    # Fallback explicit list matching HF repository structure before download
    TRAIN_FILES = [
        os.path.join(TRAIN_DIR, "train_supreme_court.json"),
        os.path.join(TRAIN_DIR, "train_high_court.json"),
        os.path.join(TRAIN_DIR, "train_statutes.json"),
        os.path.join(TRAIN_DIR, "train_tribunals.json"),
    ]

print(f"Discovered {len(TRAIN_FILES)} training file(s):")

for f in TRAIN_FILES:
    print(f" - {f}")
# -----------------------------------------------------------------------------
# Global Reproducibility & Device Config
# -----------------------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = torch.cuda.is_available()

# -----------------------------------------------------------------------------
# Hyperparameters: Model 2 (Encoder-Decoder / Seq2Seq)
# -----------------------------------------------------------------------------
TOTAL_TARGET_TOKENS = 1_000_000_000  # 1 Billion-token budget: ONE pass over it, LR anneals to MIN_LR exactly at the end
VAL_RATIO = 0.01

VOCAB_SIZE = 32000
D_MODEL = 768
N_ENC_LAYERS = 6
N_DEC_LAYERS = 6
N_HEAD = 12
D_FF = 3072

ENC_BLOCK_SIZE = 1024  # Encoder context window
DEC_BLOCK_SIZE = 1024  # Decoder target sequence window
TOTAL_BLOCK_SIZE = ENC_BLOCK_SIZE + DEC_BLOCK_SIZE

MICRO_BATCH_SIZE = 8         # samples per GPU per forward/backward (use 4 if you hit CUDA OOM)
GLOBAL_BATCH_SAMPLES = 64    # samples per optimizer step across ALL GPUs; grad-accumulation is derived from it
LEARNING_RATE = 5e-4
MIN_LR = 5e-5
WEIGHT_DECAY = 0.1
WARMUP_FRACTION = 0.05       # warm-up = 5% of the total steps ...
MAX_WARMUP_STEPS = 1000      # ... but never more than this
# NOTE: MAX_TRAIN_STEPS is no longer hard-coded. One pass over the tokenized corpus =
# len(train_dataset) // GLOBAL_BATCH_SAMPLES; it is computed in section 7.

SMALL_INIT = True            # GPT-style N(0, 0.02) init (the default N(0,1) embedding init + tied output head
                             # gives a huge initial loss). Set False to reproduce the old initialisation.

# Session / checkpoint policy (Kaggle sessions are time-limited)
SESSION_BUDGET_HOURS = 8.0   # training saves a checkpoint and stops cleanly this many hours after the notebook
                             # started (keep >= 1h below your session cap; raise it if your sessions allow ~12h)
CKPT_EVERY = 250             # optimizer steps between resumable checkpoints
EVAL_EVERY = 500             # optimizer steps between validation runs
LOG_EVERY = 50
VAL_SAMPLES = 400            # fixed validation subset (samples of 2,048 tokens)
DATALOADER_WORKERS = 2       # per GPU process
BYTES_PER_TOKEN = 4.0        # refined from real data after the tokenizer is trained

# -----------------------------------------------------------------------------
# 3. Reproducibility & Seed Initialization
# -----------------------------------------------------------------------------
SEED = 42

def set_seed(seed: int = 42):
    """Ensures complete reproducibility across all libraries."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Enable deterministic algorithms where possible
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

# -----------------------------------------------------------------------------
# 4. Output & Checkpoint Directories
# -----------------------------------------------------------------------------
OUTPUT_DIR = "/kaggle/working/cs781_phase1"
INPUT_ROOT = "/kaggle/input"
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
MODEL_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "legal-gpt2-final")

BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "best_model.pt")
TOKENIZER_PATH = os.path.join(OUTPUT_DIR, "legal_bpe.json")
TOKENIZATION_MARKER = os.path.join(OUTPUT_DIR, "tokenization_complete.json")
CONFIG_PATH = os.path.join(OUTPUT_DIR, "train_config.json")
STATUS_PATH = os.path.join(OUTPUT_DIR, "train_status.json")

PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "training_validation_loss.png")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)

# -----------------------------------------------------------------------------
# 5. Cross-session helpers
#    Kaggle wipes /kaggle/working between sessions. To continue in a new session, attach the previous
#    notebook version's OUTPUT as an input; artifacts are then found automatically (no hard-coded paths).
# -----------------------------------------------------------------------------
def find_artifact(pattern):
    """First match of `pattern` in this session's working dir, else anywhere under /kaggle/input."""
    for root in (OUTPUT_DIR, INPUT_ROOT):
        hits = sorted(glob.glob(os.path.join(root, "**", pattern), recursive=True))
        if hits:
            return hits[0]
    return None

# Session clock (used for the clean-stop deadline of the training run)
try:
    import psutil
    SESSION_START = psutil.Process(os.getpid()).create_time()
except Exception:
    SESSION_START = time.time()

print("\nConfiguration & session helpers successfully loaded.")


# 4. BPE Tokenizer training

In [ ]:
import os
import glob
import json
from huggingface_hub import hf_hub_download

# -----------------------------------------------------------------------------
# Reuse the tokenized corpus + tokenizer from a previous session if they exist
# (in /kaggle/working or in an attached previous-notebook output).
# -----------------------------------------------------------------------------
_marker = find_artifact("tokenization_complete.json")
_prev_dir = os.path.dirname(_marker) if _marker else None
if (_prev_dir
        and os.path.exists(os.path.join(_prev_dir, "legal_bpe.json"))
        and glob.glob(os.path.join(_prev_dir, "train_part_*.bin"))
        and glob.glob(os.path.join(_prev_dir, "val_part_*.bin"))):
    NEED_TOKENIZE = False
    TOKENIZED_DIR = _prev_dir
else:
    NEED_TOKENIZE = True
    TOKENIZED_DIR = OUTPUT_DIR

train_file_paths = []
if NEED_TOKENIZE:
    print("Downloading training files from Hugging Face...")
    for rel_path in TRAIN_FILES:
        try:
            downloaded_path = hf_hub_download(
                repo_id=HF_DATASET_REPO,
                filename=rel_path,
                repo_type="dataset",
                token=HF_TOKEN
            )
            train_file_paths.append(downloaded_path)
            print(f"  [+] Downloaded: {rel_path}")
        except Exception as e:
            print(f"  [-] Failed to download {rel_path}: {e}")
else:
    print(f"Found a finished tokenized corpus + tokenizer in {TOKENIZED_DIR} -> skipping the training-file download.")

print("\nDownloading test file...")
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

if NEED_TOKENIZE:
    print(f"\nSuccessfully downloaded {len(train_file_paths)} training file(s):")
    for path in train_file_paths:
        print(f" - {path}")

print(f"\nTest file downloaded to:\n - {test_file_path}")


In [ ]:
!pip install -q ijson

import os
import gc
import ijson
import random
from tokenizers import ByteLevelBPETokenizer, Tokenizer

TARGET_VOCAB_SIZE = VOCAB_SIZE   # 32,000 (token ids must stay < 65,536 because the corpus is stored as uint16)

if not NEED_TOKENIZE:
    # ---- Reuse the tokenizer that produced the existing .bin files (must match them exactly) ----
    tokenizer = Tokenizer.from_file(os.path.join(TOKENIZED_DIR, "legal_bpe.json"))
    print(f"Loaded existing tokenizer (vocab {tokenizer.get_vocab_size():,}) from {TOKENIZED_DIR}")
else:
    # -------------------------------------------------------------------------
    # Extract Representative Sample Across All Files without loading full JSONs
    # -------------------------------------------------------------------------
    SAMPLE_TARGET_BYTES = 200 * 1024 * 1024  # 200 MB text sample
    bytes_per_file = SAMPLE_TARGET_BYTES // max(len(train_file_paths), 1)

    sampled_texts = []
    total_bytes_collected = 0

    print(f"Streaming ~{SAMPLE_TARGET_BYTES / (1024 * 1024):.0f} MB of raw text for tokenizer training...")

    random.seed(SEED)

    for file_path in train_file_paths:
        file_bytes = 0
        file_name = os.path.basename(file_path)

        with open(file_path, "r", encoding="utf-8") as f:
            # ijson.items streams array elements one at a time without loading full file
            for record in ijson.items(f, "item"):
                text = record.get("text", "")
                if not text:
                    continue

                # Random sampling filter so we don't only take text from file head
                if random.random() < 0.1:
                    sampled_texts.append(text)
                    text_bytes = len(text.encode("utf-8"))
                    file_bytes += text_bytes
                    total_bytes_collected += text_bytes

                    if file_bytes >= bytes_per_file:
                        break

        gc.collect()
        print(f"  [+] Sampled {file_bytes / (1024 * 1024):.2f} MB from {file_name}")

    print(f"\nCorpus Extraction Complete: {len(sampled_texts):,} docs | {total_bytes_collected / (1024 * 1024):.2f} MB total.")

    # -------------------------------------------------------------------------
    # Train Custom Byte-Level BPE Tokenizer
    # -------------------------------------------------------------------------
    print(f"\nTraining ByteLevelBPETokenizer (Target Vocab Size: {TARGET_VOCAB_SIZE:,})...")
    tokenizer = ByteLevelBPETokenizer()
    tokenizer.train_from_iterator(sampled_texts, vocab_size=TARGET_VOCAB_SIZE)
    assert tokenizer.get_vocab_size() <= VOCAB_SIZE, "tokenizer vocab is larger than the model's embedding table"

    # Save it: the .bin files are only meaningful together with THIS tokenizer, and the notebook may restart.
    tokenizer.save(TOKENIZER_PATH)
    print(f"Saved tokenizer -> {TOKENIZER_PATH}  (vocab {tokenizer.get_vocab_size():,})")

    # Measure real bytes/token on a random probe so the corpus sub-sampling rate below is accurate.
    probe = random.sample(sampled_texts, min(2000, len(sampled_texts)))
    probe_bytes = sum(len(t.encode("utf-8")) for t in probe)
    probe_tokens = sum(len(e.ids) for e in tokenizer.encode_batch(probe))
    BYTES_PER_TOKEN = probe_bytes / max(probe_tokens, 1)
    print(f"Measured {BYTES_PER_TOKEN:.2f} bytes/token on a {len(probe):,}-document probe.")

    # Clear sample text memory
    del sampled_texts, probe
    gc.collect()
    print("Tokenizer training complete. RAM cleared.")


# 5. Dataset Tokenization

In [ ]:
import os
import glob
import json
import time
import shutil
import numpy as np
import ijson

os.environ["TOKENIZERS_PARALLELISM"] = "true"   # encode_batch is multi-threaded in Rust -> no multiprocessing needed

# -----------------------------------------------------------------------------
# Streaming tokenization.
# The old version did json.load() of the 6.5 GB file inside mp.Pool workers (~2x file size in RAM per worker);
# an OOM-killed worker makes Pool.starmap wait forever. Here ONE process streams records with ijson
# (constant memory), tokenizes them in small batches with the multi-threaded Rust tokenizer, and writes
# the uint16 .bin files incrementally, with progress logging.
# -----------------------------------------------------------------------------
def tokenize_streaming(tokenizer, file_paths, out_dir, val_ratio=0.01, target_tokens=1_000_000_000,
                       keep_prob=1.0, docs_per_batch=1000, seed=42, log_every=20):
    os.makedirs(out_dir, exist_ok=True)

    # remove partial files from an earlier failed run (the Dataset globs them!)
    for p in glob.glob(os.path.join(out_dir, "train_part_*.bin")) + glob.glob(os.path.join(out_dir, "val_part_*.bin")):
        os.remove(p)

    rng = np.random.default_rng(seed)
    f_train = open(os.path.join(out_dir, "train_part_0.bin"), "wb")
    f_val = open(os.path.join(out_dir, "val_part_0.bin"), "wb")
    n_train = n_val = n_docs = n_batches = 0
    t0 = time.time()
    batch = []

    def flush():
        nonlocal n_train, n_val, n_docs, n_batches
        if not batch:
            return
        for enc in tokenizer.encode_batch(batch):
            arr = np.asarray(enc.ids, dtype=np.uint16)
            if rng.random() < val_ratio:
                f_val.write(arr.tobytes()); n_val += arr.size
            else:
                f_train.write(arr.tobytes()); n_train += arr.size
        n_docs += len(batch)
        n_batches += 1
        batch.clear()
        if n_batches % log_every == 0:
            el = time.time() - t0
            tot = n_train + n_val
            print(f"[tok] {n_docs:,} docs | {tot/1e6:,.1f}M tokens | {tot/el/1e3:,.0f}k tok/s | {el/60:.1f} min", flush=True)

    try:
        done = False
        for path in file_paths:
            print(f"[tok] streaming {os.path.basename(path)}", flush=True)
            with open(path, "rb") as f:                       # binary mode for ijson
                for rec in ijson.items(f, "item"):            # one record at a time
                    text = rec.get("text")
                    if not text:
                        continue
                    # uniform random sub-sampling spreads the token budget over the WHOLE corpus
                    # (instead of only taking the first N GB of each file)
                    if keep_prob < 1.0 and rng.random() >= keep_prob:
                        continue
                    batch.append(text)
                    if len(batch) >= docs_per_batch:
                        flush()
                        if n_train + n_val >= target_tokens:
                            done = True
                            break
            if done:
                break
        flush()
    finally:
        f_train.close(); f_val.close()

    print(f"[tok] DONE  train={n_train:,}  val={n_val:,}  total={n_train+n_val:,} tokens in {(time.time()-t0)/60:.1f} min")
    return n_train, n_val, n_docs


if NEED_TOKENIZE:
    print(f"ijson backend: {ijson.backend}  (should be a yajl2_c variant; 'python' would be slow)")
    est_total_tokens = sum(os.path.getsize(p) for p in train_file_paths) / BYTES_PER_TOKEN
    KEEP_PROB = min(1.0, 1.08 * TOTAL_TARGET_TOKENS / est_total_tokens)   # ~8% head-room, stop at the target anyway
    print(f"Estimated corpus size: {est_total_tokens/1e9:.2f}B tokens -> keeping {KEEP_PROB:.1%} of documents "
          f"to reach a {TOTAL_TARGET_TOKENS/1e9:.1f}B-token budget.")

    n_train, n_val, n_docs = tokenize_streaming(
        tokenizer, train_file_paths, OUTPUT_DIR,
        val_ratio=VAL_RATIO, target_tokens=TOTAL_TARGET_TOKENS, keep_prob=KEEP_PROB, seed=SEED,
    )
    with open(TOKENIZATION_MARKER, "w") as f:   # written LAST: its presence means "tokenization finished"
        json.dump({"train_tokens": n_train, "val_tokens": n_val, "docs": n_docs, "vocab_size": tokenizer.get_vocab_size(),
                   "bytes_per_token": BYTES_PER_TOKEN, "keep_prob": KEEP_PROB}, f)
    TOKENIZED_DIR = OUTPUT_DIR
else:
    with open(os.path.join(TOKENIZED_DIR, "tokenization_complete.json")) as f:
        _m = json.load(f)
    print(f"Reusing tokenized corpus from {TOKENIZED_DIR}: {_m['train_tokens']:,} train / {_m['val_tokens']:,} val tokens.")
    if os.path.abspath(TOKENIZED_DIR) != os.path.abspath(OUTPUT_DIR):
        # copy forward so THIS version's output is self-contained (the next session then only needs this one attached)
        for pat in ("train_part_*.bin", "val_part_*.bin", "legal_bpe.json", "tokenization_complete.json"):
            for src in glob.glob(os.path.join(TOKENIZED_DIR, pat)):
                dst = os.path.join(OUTPUT_DIR, os.path.basename(src))
                if not os.path.exists(dst):
                    shutil.copy2(src, dst)
        TOKENIZED_DIR = OUTPUT_DIR
        print("Copied tokenized corpus + tokenizer into /kaggle/working so they persist with this version.")


# 6. Encoder-Decoder Transformer Model

In [ ]:
%%writefile /kaggle/working/cs781_phase1/model_seq2seq.py
import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        var = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(var + self.eps) * self.weight


def causal_mask_like(S: int, device) -> torch.Tensor:
    return torch.triu(torch.full((S, S), float("-inf"), device=device), diagonal=1)


class EncoderBlock(nn.Module):
    """Bidirectional Encoder Block."""
    def __init__(self, d_model: int, n_head: int, d_ff: int):
        super().__init__()
        self.ln1 = RMSNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_head, batch_first=True)
        self.ln2 = RMSNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff, bias=False),
            nn.GELU(),
            nn.Linear(d_ff, d_model, bias=False),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        norm_x = self.ln1(x)
        # need_weights=False -> fused scaled-dot-product-attention kernel (no S x S map is materialised)
        attn_out, _ = self.attn(norm_x, norm_x, norm_x, need_weights=False)
        x = x + attn_out
        x = x + self.mlp(self.ln2(x))
        return x


class DecoderBlock(nn.Module):
    """Causal Decoder Block with Encoder Cross-Attention."""
    def __init__(self, d_model: int, n_head: int, d_ff: int):
        super().__init__()
        self.ln1 = RMSNorm(d_model)
        self.self_attn = nn.MultiheadAttention(d_model, n_head, batch_first=True)
        self.ln2 = RMSNorm(d_model)
        self.cross_attn = nn.MultiheadAttention(d_model, n_head, batch_first=True)
        self.ln3 = RMSNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff, bias=False),
            nn.GELU(),
            nn.Linear(d_ff, d_model, bias=False),
        )

    def forward(self, x: torch.Tensor, enc_out: torch.Tensor, causal_mask: torch.Tensor = None) -> torch.Tensor:
        if causal_mask is None:
            causal_mask = causal_mask_like(x.size(1), x.device)
        # 1. Causal self-attention (is_causal hint + need_weights=False -> fused causal kernel)
        norm_x = self.ln1(x)
        self_attn_out, _ = self.self_attn(
            norm_x, norm_x, norm_x, attn_mask=causal_mask, is_causal=True, need_weights=False
        )
        x = x + self_attn_out

        # 2. Cross-attention over encoder hidden states
        norm_x2 = self.ln2(x)
        cross_attn_out, _ = self.cross_attn(
            query=norm_x2, key=enc_out, value=enc_out, need_weights=False
        )
        x = x + cross_attn_out

        # 3. Feed-forward MLP
        x = x + self.mlp(self.ln3(x))
        return x


class Seq2SeqLegalTransformer(nn.Module):
    """Full Encoder-Decoder Transformer Architecture."""
    def __init__(self, vocab_size: int, d_model: int, n_enc_layers: int, n_dec_layers: int,
                 n_head: int, d_ff: int, max_len: int = 2048, small_init: bool = True):
        super().__init__()
        self.d_model = d_model
        self.shared_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Parameter(torch.zeros(1, max_len, d_model))

        self.encoder_layers = nn.ModuleList([EncoderBlock(d_model, n_head, d_ff) for _ in range(n_enc_layers)])
        self.decoder_layers = nn.ModuleList([DecoderBlock(d_model, n_head, d_ff) for _ in range(n_dec_layers)])

        self.enc_ln = RMSNorm(d_model)
        self.dec_ln = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        if small_init:
            # nn.Embedding defaults to N(0,1); with tied output weights that gives logits of std ~sqrt(d_model)
            # and a huge initial loss. GPT-style N(0, 0.02) init avoids that.
            self.apply(self._init_weights)
            nn.init.normal_(self.pos_embed, mean=0.0, std=0.02)

        # Weight tying (after init so the shared tensor keeps the small init)
        self.lm_head.weight = self.shared_embed.weight

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def encode(self, enc_input_ids: torch.Tensor) -> torch.Tensor:
        S = enc_input_ids.size(1)
        x = self.shared_embed(enc_input_ids) + self.pos_embed[:, :S, :]
        for layer in self.encoder_layers:
            x = layer(x)
        return self.enc_ln(x)

    def decode(self, dec_input_ids: torch.Tensor, enc_out: torch.Tensor) -> torch.Tensor:
        S = dec_input_ids.size(1)
        x = self.shared_embed(dec_input_ids) + self.pos_embed[:, :S, :]
        causal_mask = causal_mask_like(S, dec_input_ids.device)
        for layer in self.decoder_layers:
            x = layer(x, enc_out, causal_mask=causal_mask)
        return self.dec_ln(x)

    def forward(self, enc_input_ids: torch.Tensor, dec_input_ids: torch.Tensor, dec_target_ids: torch.Tensor = None):
        enc_out = self.encode(enc_input_ids)
        dec_out = self.decode(dec_input_ids, enc_out)
        logits = self.lm_head(dec_out)

        loss = None
        if dec_target_ids is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), dec_target_ids.view(-1))
        return logits, loss


In [ ]:
import sys
import importlib
sys.path.insert(0, OUTPUT_DIR)
import model_seq2seq
importlib.reload(model_seq2seq)
from model_seq2seq import Seq2SeqLegalTransformer

MODEL_KWARGS = dict(vocab_size=VOCAB_SIZE, d_model=D_MODEL, n_enc_layers=N_ENC_LAYERS, n_dec_layers=N_DEC_LAYERS,
                    n_head=N_HEAD, d_ff=D_FF, max_len=2048, small_init=SMALL_INIT)

# Build once on the CPU just to report the size (the GPUs are not touched; training builds the real model per GPU).
_probe = Seq2SeqLegalTransformer(**MODEL_KWARGS)
print(f"Model 2 (Seq2Seq Transformer) defined. Parameters: {sum(p.numel() for p in _probe.parameters()) / 1e6:.1f}M")
del _probe
gc.collect()


# 7. Dataloader

In [ ]:
%%writefile /kaggle/working/cs781_phase1/data_seq2seq.py
import glob
import numpy as np
import torch
from torch.utils.data import Dataset


class Seq2SeqMemmapDataset(Dataset):
    """Disk-backed uint16 token stream, cut into (encoder window | decoder window) samples."""
    def __init__(self, file_pattern: str, enc_size: int, dec_size: int):
        self.enc_size = enc_size
        self.dec_size = dec_size
        self.total_seq_len = enc_size + dec_size

        import os
        self.bin_files = [f for f in sorted(glob.glob(file_pattern)) if os.path.getsize(f) > 0]
        self.mmaps = [np.memmap(f, dtype=np.uint16, mode="r") for f in self.bin_files]
        self.lengths = [len(m) for m in self.mmaps]
        self.total_tokens = sum(self.lengths)
        self.cum_lengths = np.cumsum([0] + self.lengths)
        self.num_samples = max(0, (self.total_tokens - 1) // self.total_seq_len)

    def __len__(self) -> int:
        return self.num_samples

    def __getitem__(self, idx: int):
        start_pos = int(idx) * self.total_seq_len
        file_idx = int(np.searchsorted(self.cum_lengths, start_pos, side="right") - 1)
        local_start = start_pos - int(self.cum_lengths[file_idx])
        local_end = local_start + self.total_seq_len

        if local_end <= self.lengths[file_idx]:
            chunk = self.mmaps[file_idx][local_start:local_end].astype(np.int64)
        else:
            chunk = self.mmaps[file_idx][local_start:].astype(np.int64)
            rem = self.total_seq_len - len(chunk)
            if file_idx + 1 < len(self.mmaps):
                chunk = np.concatenate([chunk, self.mmaps[file_idx + 1][:rem].astype(np.int64)])

        enc_inputs = torch.from_numpy(chunk[: self.enc_size])
        dec_inputs = torch.from_numpy(chunk[self.enc_size: -1])
        dec_targets = torch.from_numpy(chunk[self.enc_size + 1:])
        return enc_inputs, dec_inputs, dec_targets


def rank_micro_batches(num_samples, world_size, rank, micro_bs, accum, seed, start_step=0, max_steps=None):
    """
    Deterministic, resumable data order (one pass, no repeats).
    Optimizer step s always consumes perm[s*G:(s+1)*G] with G = world*micro_bs*accum, so a resumed run
    (even on a different number of GPUs, if G is kept constant) continues exactly where it stopped.
    Returns (total_steps, list_of_index_lists) for this rank, starting at `start_step`.
    """
    G = world_size * micro_bs * accum
    total_steps = num_samples // G
    n_steps = total_steps if max_steps is None else min(total_steps, max_steps)
    perm = np.random.default_rng(seed).permutation(num_samples)[: total_steps * G]
    perm = perm.reshape(total_steps, accum, world_size, micro_bs)[:n_steps]
    mine = perm[start_step:, :, rank, :].reshape(-1, micro_bs)
    return total_steps, mine.tolist()


def val_micro_batches(num_samples, world_size, rank, micro_bs, n_total):
    """Fixed random subset of the validation set, split across ranks."""
    per = world_size * micro_bs
    n = (min(num_samples, n_total) // per) * per
    if n == 0:
        return []
    idx = np.random.default_rng(0).permutation(num_samples)[:n].reshape(-1, world_size, micro_bs)
    return idx[:, rank, :].tolist()


In [ ]:
import sys
sys.path.insert(0, OUTPUT_DIR)
import importlib
import data_seq2seq
importlib.reload(data_seq2seq)
from data_seq2seq import Seq2SeqMemmapDataset

train_dataset = Seq2SeqMemmapDataset(os.path.join(TOKENIZED_DIR, "train_part_*.bin"), ENC_BLOCK_SIZE, DEC_BLOCK_SIZE)
val_dataset = Seq2SeqMemmapDataset(os.path.join(TOKENIZED_DIR, "val_part_*.bin"), ENC_BLOCK_SIZE, DEC_BLOCK_SIZE)

# One pass over the whole tokenized corpus; the cosine schedule is built on this number, so the learning rate
# anneals to MIN_LR exactly when the data runs out.
MAX_TRAIN_STEPS = len(train_dataset) // GLOBAL_BATCH_SAMPLES
WARMUP_STEPS = int(max(10, min(MAX_WARMUP_STEPS, WARMUP_FRACTION * MAX_TRAIN_STEPS)))
TOKENS_PER_STEP = GLOBAL_BATCH_SAMPLES * TOTAL_BLOCK_SIZE

print(f"Dataset Loaded: {len(train_dataset):,} Training Samples | {len(val_dataset):,} Val Samples.")
print(f"Train tokens: {train_dataset.total_tokens/1e6:,.1f}M  |  tokens/optimizer step: {TOKENS_PER_STEP:,}")
print(f"MAX_TRAIN_STEPS = {MAX_TRAIN_STEPS:,} (one full pass)  |  warm-up = {WARMUP_STEPS} steps")
print("Wall-clock is printed during training (s/step and ETA); with fused attention on 2xT4 expect a few hours per pass.")


# 8. Training Loop

Training script (written to disk and launched with `torchrun` on both T4s; resumable, time-boxed).

In [ ]:
%%writefile /kaggle/working/cs781_phase1/train_ddp.py
"""
Resumable DDP pre-training script for the encoder-decoder legal LM.
Launched by the notebook with:  python -m torch.distributed.run --standalone --nproc_per_node=<GPUs> train_ddp.py config.json
Works with 1 or 2 GPUs (global batch is fixed, grad-accumulation adapts) and on CPU (gloo) for smoke tests.
"""
import os, sys, json, time, math, glob, contextlib
import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader

cfg = json.load(open(sys.argv[1]))
sys.path.insert(0, cfg["output_dir"])
from model_seq2seq import Seq2SeqLegalTransformer
from data_seq2seq import Seq2SeqMemmapDataset, rank_micro_batches, val_micro_batches

# ----------------------------------------------------------------------------- distributed setup
rank = int(os.environ.get("RANK", 0))
world = int(os.environ.get("WORLD_SIZE", 1))
local_rank = int(os.environ.get("LOCAL_RANK", 0))
use_cuda = torch.cuda.is_available()
if use_cuda:
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)
else:
    device = torch.device("cpu")
if world > 1:
    dist.init_process_group(backend="nccl" if use_cuda else "gloo")
is_main = rank == 0
use_amp = use_cuda


def log(*a):
    if is_main:
        print(*a, flush=True)


torch.manual_seed(cfg["seed"])
np.random.seed(cfg["seed"])

# ----------------------------------------------------------------------------- hyper-parameters
micro_bs = cfg["micro_batch_size"]
gbs = cfg["global_batch_samples"]
assert gbs % (world * micro_bs) == 0, f"global batch {gbs} not divisible by world({world})*micro({micro_bs})"
accum = gbs // (world * micro_bs)
enc_size, dec_size = cfg["enc_block_size"], cfg["dec_block_size"]
tokens_per_step = gbs * (enc_size + dec_size)

train_ds = Seq2SeqMemmapDataset(os.path.join(cfg["tokenized_dir"], "train_part_*.bin"), enc_size, dec_size)
val_ds = Seq2SeqMemmapDataset(os.path.join(cfg["tokenized_dir"], "val_part_*.bin"), enc_size, dec_size)
total_steps = len(train_ds) // gbs
max_steps = total_steps if cfg.get("max_train_steps") is None else min(total_steps, int(cfg["max_train_steps"]))
warmup = int(max(10, min(cfg["max_warmup_steps"], cfg["warmup_fraction"] * max_steps)))
lr_max, lr_min = cfg["learning_rate"], cfg["min_lr"]


def get_lr(step):
    if step < warmup:
        return lr_max * (step + 1) / warmup
    if step >= max_steps:
        return lr_min
    ratio = (step - warmup) / max(1, max_steps - warmup)
    return lr_min + 0.5 * (1.0 + math.cos(math.pi * ratio)) * (lr_max - lr_min)


ckpt_dir = cfg["checkpoint_dir"]
os.makedirs(ckpt_dir, exist_ok=True)
latest_path = os.path.join(ckpt_dir, "latest_checkpoint.pt")
best_path = os.path.join(ckpt_dir, "best_model.pt")
status_path = cfg["status_path"]

log(f"[setup] world={world} device={device} micro_bs={micro_bs} accum={accum} global_batch={gbs} samples "
    f"({tokens_per_step:,} tokens/step)")
log(f"[setup] train samples={len(train_ds):,} val samples={len(val_ds):,} -> {max_steps:,} optimizer steps "
    f"(one pass), warmup={warmup}")

# ----------------------------------------------------------------------------- model / optimizer
model = Seq2SeqLegalTransformer(**cfg["model"]).to(device)
raw_model = model
opt = torch.optim.AdamW(raw_model.parameters(), lr=lr_max, weight_decay=cfg["weight_decay"], betas=(0.9, 0.95))
try:
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


def pick_resume():
    """Newest checkpoint (highest step) in the working dir or in any attached input (previous notebook output)."""
    cands = []
    for root in cfg["resume_roots"]:
        cands += glob.glob(os.path.join(root, "**", "latest_checkpoint.pt"), recursive=True)
    best_p, best_s = None, -1
    for p in sorted(set(cands)):
        try:
            try:
                m = torch.load(p, map_location="cpu", mmap=True, weights_only=False)
            except TypeError:
                m = torch.load(p, map_location="cpu")
            s = int(m.get("step", -1))
            del m
        except Exception as e:  # corrupt / half-written file
            log(f"[resume] skipping unreadable checkpoint {p}: {e}")
            continue
        if s > best_s:
            best_p, best_s = p, s
    return best_p, best_s


global_step = 0
best_val = float("inf")
resume_path, resume_step = pick_resume()
if resume_path is not None:
    ck = torch.load(resume_path, map_location="cpu", weights_only=False)
    raw_model.load_state_dict(ck["model_state"])
    opt.load_state_dict(ck["optimizer_state"])
    if use_amp and ck.get("scaler_state"):
        scaler.load_state_dict(ck["scaler_state"])
    global_step = int(ck["step"])
    if os.path.exists(best_path):
        best_val = float(ck.get("best_val_loss", float("inf")))
    log(f"[resume] restored step {global_step:,} from {resume_path} (best val so far: {best_val:.4f})")
    del ck
else:
    log("[resume] no checkpoint found -> starting from scratch")

if world > 1:
    model = DDP(model, device_ids=[local_rank] if use_cuda else None)
n_params = sum(p.numel() for p in raw_model.parameters())
log(f"[model] {n_params/1e6:.1f}M parameters")


# ----------------------------------------------------------------------------- helpers
def write_status(finished):
    if not is_main:
        return
    tmp = status_path + ".tmp"
    with open(tmp, "w") as f:
        json.dump({"finished": bool(finished), "step": global_step, "max_steps": max_steps,
                   "best_val_loss": None if best_val == float("inf") else best_val,
                   "updated": time.time()}, f)
    os.replace(tmp, status_path)


def save_checkpoint():
    if not is_main:
        return
    ck = {"model_state": raw_model.state_dict(), "optimizer_state": opt.state_dict(),
          "scaler_state": scaler.state_dict() if use_amp else None,
          "step": global_step, "best_val_loss": best_val, "tokens_seen": global_step * tokens_per_step}
    tmp = latest_path + ".tmp"
    torch.save(ck, tmp)
    os.replace(tmp, latest_path)
    log(f"[ckpt] saved step {global_step:,} -> {latest_path}")


@torch.no_grad()
def evaluate():
    raw_model.eval()
    batches = val_micro_batches(len(val_ds), world, rank, micro_bs, cfg["val_samples"])
    tot = torch.zeros(2, device=device)
    if batches:
        for enc, dec_in, tgt in DataLoader(val_ds, batch_sampler=batches, num_workers=0):
            enc, dec_in, tgt = enc.to(device), dec_in.to(device), tgt.to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                _, l = raw_model(enc, dec_in, tgt)
            tot[0] += l.float()
            tot[1] += 1
    if world > 1:
        dist.all_reduce(tot)
    raw_model.train()
    return (tot[0] / tot[1]).item() if tot[1] > 0 else float("nan")


# ----------------------------------------------------------------------------- training loop
t_start = time.time()
if global_step >= max_steps:
    log("[train] checkpoint is already at the final step - nothing left to train.")
    write_status(True)
else:
    _, train_batches = rank_micro_batches(len(train_ds), world, rank, micro_bs, accum, cfg["seed"],
                                          start_step=global_step, max_steps=max_steps)
    loader = DataLoader(train_ds, batch_sampler=train_batches, num_workers=cfg["num_workers"], pin_memory=use_cuda)
    it = iter(loader)
    model.train()
    opt.zero_grad(set_to_none=True)
    t_last, steps_since_log = time.time(), 0
    finished = False
    log(f"[train] starting at step {global_step:,}/{max_steps:,}")

    while global_step < max_steps:
        lr = get_lr(global_step)
        for g in opt.param_groups:
            g["lr"] = lr

        loss_sum = torch.zeros((), device=device)
        for micro in range(accum):
            enc, dec_in, tgt = next(it)
            enc, dec_in, tgt = enc.to(device, non_blocking=True), dec_in.to(device, non_blocking=True), tgt.to(device, non_blocking=True)
            sync_ctx = contextlib.nullcontext() if (world == 1 or micro == accum - 1) else model.no_sync()
            with sync_ctx:
                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                    _, loss = model(enc, dec_in, tgt)
                scaler.scale(loss / accum).backward()
            loss_sum += loss.detach().float()
            del loss, _

        scaler.unscale_(opt)
        gnorm = torch.nn.utils.clip_grad_norm_(raw_model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)
        global_step += 1
        steps_since_log += 1

        if global_step % cfg["log_every"] == 0 or global_step == 1:
            avg = loss_sum / accum
            if world > 1:
                dist.all_reduce(avg)
                avg = avg / world
            v = avg.item()
            if not math.isfinite(v):
                log(f"[train] NON-FINITE loss at step {global_step} - aborting (last good checkpoint is kept).")
                sys.exit(2)
            now = time.time()
            sps = (now - t_last) / max(1, steps_since_log)
            eta_h = sps * (max_steps - global_step) / 3600
            mem = f" | mem {torch.cuda.max_memory_allocated()/2**30:.1f}GB" if use_cuda else ""
            log(f"step {global_step:6d}/{max_steps} | loss {v:.4f} | lr {lr:.2e} | gnorm {gnorm.item():.2f} | "
                f"{sps:.2f}s/step | {tokens_per_step/sps/1e3:.1f}k tok/s | elapsed {(now-t_start)/3600:.2f}h | "
                f"ETA {eta_h:.2f}h{mem}")
            t_last, steps_since_log = now, 0

        stop = False
        if global_step % 10 == 0:
            flag = torch.tensor([1.0 if time.time() > cfg["deadline_epoch"] else 0.0], device=device)
            if world > 1:
                dist.all_reduce(flag, op=dist.ReduceOp.MAX)
            stop = flag.item() > 0

        do_eval = (global_step % cfg["eval_every"] == 0) or global_step == max_steps
        if do_eval:
            val = evaluate()
            ppl = math.exp(val) if val < 20 else float("inf")
            log(f"\n[val] step {global_step} | val loss {val:.4f} | ppl {ppl:.2f}\n")
            if is_main and val == val and val < best_val:  # val == val -> not NaN
                best_val = val
                tmp = best_path + ".tmp"
                torch.save({"model_state": raw_model.state_dict(), "step": global_step, "best_val_loss": best_val}, tmp)
                os.replace(tmp, best_path)
                log(f"[val] new best -> {best_path}")
            if world > 1:  # all ranks must agree on best_val for the checkpoint they might write
                bv = torch.tensor([best_val if best_val != float("inf") else 1e30], device=device)
                dist.broadcast(bv, src=0)
                best_val = float(bv.item()) if bv.item() < 1e29 else float("inf")

        if do_eval or stop or global_step % cfg["ckpt_every"] == 0:
            save_checkpoint()
        if global_step == max_steps:
            finished = True
        if stop and not finished:
            log(f"[train] session time budget reached at step {global_step:,} - checkpoint saved, stopping cleanly.")
            break

    write_status(finished)
    log(f"[train] {'FINISHED' if finished else 'PAUSED'} at step {global_step:,}/{max_steps:,} | best val {best_val:.4f}")

if world > 1:
    dist.barrier()
    dist.destroy_process_group()


In [ ]:
import os
import sys
import json
import time
import subprocess

NUM_GPUS = torch.cuda.device_count()
assert NUM_GPUS >= 1, "No GPU found - enable 'GPU T4 x2' under Session options -> Accelerator."
assert GLOBAL_BATCH_SAMPLES % (NUM_GPUS * MICRO_BATCH_SIZE) == 0, \
    f"GLOBAL_BATCH_SAMPLES ({GLOBAL_BATCH_SAMPLES}) must be divisible by NUM_GPUS*MICRO_BATCH_SIZE ({NUM_GPUS * MICRO_BATCH_SIZE})"

DEADLINE_EPOCH = SESSION_START + SESSION_BUDGET_HOURS * 3600
remaining_h = (DEADLINE_EPOCH - time.time()) / 3600
print(f"{NUM_GPUS} GPU(s) | grad-accum per GPU = {GLOBAL_BATCH_SAMPLES // (NUM_GPUS * MICRO_BATCH_SIZE)} | "
      f"time left before the clean-stop deadline: {remaining_h:.1f} h")
if remaining_h < 0.25:
    print("WARNING: almost no time left in this session's budget - training will pause right away. "
          "Start a new session / raise SESSION_BUDGET_HOURS.")

train_cfg = dict(
    seed=SEED, output_dir=OUTPUT_DIR, tokenized_dir=TOKENIZED_DIR, checkpoint_dir=CHECKPOINT_DIR,
    status_path=STATUS_PATH, resume_roots=[OUTPUT_DIR, INPUT_ROOT],
    micro_batch_size=MICRO_BATCH_SIZE, global_batch_samples=GLOBAL_BATCH_SAMPLES,
    enc_block_size=ENC_BLOCK_SIZE, dec_block_size=DEC_BLOCK_SIZE,
    learning_rate=LEARNING_RATE, min_lr=MIN_LR, weight_decay=WEIGHT_DECAY,
    warmup_fraction=WARMUP_FRACTION, max_warmup_steps=MAX_WARMUP_STEPS,
    max_train_steps=None,                      # None -> one full pass over the tokenized corpus
    num_workers=DATALOADER_WORKERS, log_every=LOG_EVERY, eval_every=EVAL_EVERY, ckpt_every=CKPT_EVERY,
    val_samples=VAL_SAMPLES, deadline_epoch=DEADLINE_EPOCH, model=MODEL_KWARGS,
)
with open(CONFIG_PATH, "w") as f:
    json.dump(train_cfg, f, indent=2)

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["TOKENIZERS_PARALLELISM"] = "false"
# env["NCCL_P2P_DISABLE"] = "1"   # uncomment ONLY if the run hangs before step 1 (NCCL peer-to-peer issue)

cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={NUM_GPUS}",
       os.path.join(OUTPUT_DIR, "train_ddp.py"), CONFIG_PATH]
print("Launching:", " ".join(cmd), "\n")

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
for line in proc.stdout:
    print(line, end="")
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"Training process exited with code {rc} - see the log above. "
                       f"The last good checkpoint (if any) is in {CHECKPOINT_DIR}.")

status = json.load(open(STATUS_PATH)) if os.path.exists(STATUS_PATH) else {}
TRAINING_DONE = bool(status.get("finished", False))
print("\nTRAINING FINISHED - run the evaluation cells below." if TRAINING_DONE else
      f"\nPAUSED at step {status.get('step')}/{status.get('max_steps')}: checkpoint saved. Evaluation is skipped; "
      "start the next session (attach this notebook's output as input) to continue.")


# 9. Perplexity 

In [ ]:
import sys
sys.path.insert(0, OUTPUT_DIR)
from model_seq2seq import Seq2SeqLegalTransformer
from tokenizers import Tokenizer

# The tokenizer lives in memory only if cells 4-5 ran in this kernel; otherwise load the saved one.
if "tokenizer" not in globals():
    tokenizer = Tokenizer.from_file(find_artifact("legal_bpe.json"))

# -----------------------------------------------------------------------------
# 2. Compute Test Perplexity
# -----------------------------------------------------------------------------
@torch.no_grad()
def evaluate_test_ppl(model, test_records, tokenizer):
    total_loss = 0.0
    total_tokens = 0

    for record in test_records:
        text = record.get("text", record.get("prompt", ""))
        if not text:
            continue

        try:
            tokens = tokenizer.encode(text).ids
        except AttributeError:
            tokens = tokenizer.encode(text)

        if len(tokens) < 4:
            continue

        tokens = tokens[: TOTAL_BLOCK_SIZE]
        half = len(tokens) // 2

        enc_ids = torch.tensor([tokens[:half]], dtype=torch.long, device=device)
        dec_in_ids = torch.tensor([tokens[half:-1]], dtype=torch.long, device=device)
        dec_target_ids = torch.tensor([tokens[half+1:]], dtype=torch.long, device=device)

        with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
            _, loss = model(enc_ids, dec_in_ids, dec_target_ids)

        seq_len = dec_target_ids.size(1)
        total_loss += loss.item() * seq_len
        total_tokens += seq_len

    if total_tokens == 0:
        return float("inf"), float("inf")

    avg_loss = total_loss / total_tokens
    try:
        ppl = math.exp(avg_loss)
    except OverflowError:
        ppl = float("inf")

    return avg_loss, ppl


# -----------------------------------------------------------------------------
# 3. Seq2Seq Text Completion Generator
# -----------------------------------------------------------------------------
@torch.no_grad()
def generate_seq2seq_completion(model, tokenizer, prompt_text, max_new_tokens=128):
    model.eval()

    try:
        prompt_ids = tokenizer.encode(prompt_text).ids
    except AttributeError:
        prompt_ids = tokenizer.encode(prompt_text)

    if not prompt_ids:
        return ""

    enc_input = torch.tensor([prompt_ids[:ENC_BLOCK_SIZE]], dtype=torch.long, device=device)
    enc_out = model.encode(enc_input)

    # Start decoder with last prompt token or first target token
    generated = [prompt_ids[-1]]

    for _ in range(max_new_tokens):
        dec_input = torch.tensor([generated[-DEC_BLOCK_SIZE:]], dtype=torch.long, device=device)
        dec_out = model.decode(dec_input, enc_out)
        logits = model.lm_head(dec_out[:, -1, :])

        next_token = torch.argmax(logits, dim=-1).item()
        generated.append(next_token)

        if hasattr(tokenizer, "eos_id") and next_token == tokenizer.eos_id:
            break

    try:
        return tokenizer.decode(generated[1:])
    except Exception:
        return ""


# -----------------------------------------------------------------------------
# 1. Only evaluate once pre-training has actually finished (a paused run just skips this cell)
# -----------------------------------------------------------------------------
_status = json.load(open(STATUS_PATH)) if os.path.exists(STATUS_PATH) else {}
TRAINING_DONE = bool(_status.get("finished", False))

if not TRAINING_DONE:
    print(f"Pre-training is not finished yet (step {_status.get('step')}/{_status.get('max_steps')}). "
          "Skipping evaluation - continue training in the next session (attach this notebook's output as input).")
else:
    # -------------------------------------------------------------------------
    # Load Best Weights (fall back to the latest checkpoint if best_model.pt is not in this session's output)
    # -------------------------------------------------------------------------
    _best = BEST_MODEL_PATH if os.path.exists(BEST_MODEL_PATH) else (find_artifact("best_model.pt") or find_artifact("latest_checkpoint.pt"))
    assert _best is not None, "No trained weights found."
    ckpt = torch.load(_best, map_location="cpu", weights_only=False)
    model = Seq2SeqLegalTransformer(**MODEL_KWARGS).to(device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    print(f"Loaded weights from {_best} (step {ckpt.get('step', 0)})")
    del ckpt

    # -----------------------------------------------------------------------------
    # 4. Generate Predictions & Export JSON
    # -----------------------------------------------------------------------------
    with open(test_file_path, "r", encoding="utf-8") as f:
        test_records = json.load(f)

    test_loss, test_ppl = evaluate_test_ppl(model, test_records, tokenizer)

    print("\n" + "=" * 50)
    print(f"  Model 2 Test Loss : {test_loss:.4f}")
    print(f"  Model 2 Test PPL  : {test_ppl:.2f}")
    print("=" * 50 + "\n")

    predictions = []
    print(f"Generating completions for {len(test_records):,} test records...")

    for idx, record in enumerate(test_records):
        prompt = record.get("prompt", record.get("text", record.get("input", "")))
        record_id = record.get("id", idx)

        completion = generate_seq2seq_completion(model, tokenizer, prompt, max_new_tokens=128)

        predictions.append({
            "id": record_id,
            "prompt": prompt,
            "prediction": completion,
            "test_ppl": round(test_ppl, 2)
        })

    with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
        json.dump(predictions, f, indent=2, ensure_ascii=False)

    print(f"Evaluation Complete! Saved predictions to -> {PREDICTION_PATH}")

# 10. Zip submission file

In [ ]:
import zipfile

# -----------------------------------------------------------------------------
# Submission Packaging: Zip Only Prediction File
# -----------------------------------------------------------------------------
SUBMISSION_ZIP_PATH = "/kaggle/working/submission_model2.zip"

if os.path.exists(PREDICTION_PATH):
    print(f"Zipping prediction file from: {PREDICTION_PATH}...")

    with zipfile.ZipFile(SUBMISSION_ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
        arcname = os.path.basename(PREDICTION_PATH)
        zipf.write(PREDICTION_PATH, arcname=arcname)

    zip_size_mb = os.path.getsize(SUBMISSION_ZIP_PATH) / (1024 ** 2)

    print("\n" + "=" * 50)
    print("Model 2 Submission Package Complete!")
    print(f"  - Archive Path : {SUBMISSION_ZIP_PATH}")
    print(f"  - Included File: {arcname}")
    print(f"  - Archive Size : {zip_size_mb:.2f} MB")
    print("=" * 50)
else:
    print(f"No prediction file at {PREDICTION_PATH} yet - it is written by the evaluation cell once pre-training has finished.")


#### ---------------------------------COMPLETE-----------------------------------------